# Hash simples: XOR bloco a bloco e XOR com rotação

Dois hashes de 64 bits feitos só com Python padrão, seguindo a mesma ideia geral do SHA-256
(padding -> blocos -> misturar cada bloco no estado), mas com operações bem mais simples.

Usamos como exemplo a mensagem `"red block blue"`, a mesma do vídeo sobre SHA-256.

## 1. Padding

Como no SHA, acrescentamos um bit `1`, depois zeros, e por fim o tamanho original da mensagem
(em bits) nos últimos 64 bits. O resultado é dividido em blocos de 64 bits (8 bytes).

In [1]:
TAMANHO_BLOCO = 8  # bytes = 64 bits


def padding(mensagem: bytes) -> bytes:
    tamanho_bits = len(mensagem) * 8
    dados = mensagem + b"\x80"  # bit 1 seguido de sete zeros
    # zeros até sobrar exatamente 8 bytes no último bloco para o tamanho
    while (len(dados) + 8) % TAMANHO_BLOCO != 0:
        dados += b"\x00"
    return dados + tamanho_bits.to_bytes(8, "big")


def blocos(dados: bytes) -> list[int]:
    return [
        int.from_bytes(dados[i : i + TAMANHO_BLOCO], "big")
        for i in range(0, len(dados), TAMANHO_BLOCO)
    ]


msg = b"red block blue"
dados = padding(msg)
print(f"mensagem : {len(msg)} bytes = {len(msg) * 8} bits")
print(f"com pad  : {len(dados)} bytes = {len(dados) * 8} bits = {len(dados) // TAMANHO_BLOCO} blocos")
for i, b in enumerate(blocos(dados)):
    print(f"bloco {i}: {b:016x}")

mensagem : 14 bytes = 112 bits
com pad  : 24 bytes = 192 bits = 3 blocos
bloco 0: 72656420626c6f63
bloco 1: 6b20626c75658000
bloco 2: 0000000000000070


## 2. Hash XOR bloco a bloco

O estado começa em zero e cada bloco é combinado com `XOR`. Simples, mas fraco: `XOR` é comutativo,
então trocar a ordem dos blocos não altera o resultado, e cada bit da entrada só mexe em um bit da saída.

In [2]:
def hash_xor(mensagem: bytes) -> str:
    estado = 0
    for bloco in blocos(padding(mensagem)):
        estado ^= bloco
    return f"{estado:016x}"


print("hash_xor('red block blue') =", hash_xor(b"red block blue"))

hash_xor('red block blue') = 1945064c1709ef13


## 3. Hash XOR com rotação

Mesma ideia, mas entre um bloco e outro o estado é rotacionado à esquerda e misturado com uma constante
(a parte fracionária da raiz quadrada de 2, como o `H0` do SHA-256). A rotação faz com que os bits de
um bloco caiam em posições diferentes dos bits do bloco seguinte, quebrando a comutatividade.

In [3]:
MASK64 = (1 << 64) - 1
CONSTANTE = 0x6A09E667F3BCC908  # parte fracionária de sqrt(2), como no SHA


def rotl(x: int, n: int) -> int:
    """Rotação de 64 bits para a esquerda."""
    n %= 64
    return ((x << n) | (x >> (64 - n))) & MASK64


def hash_xor_rot(mensagem: bytes, rodadas: int = 4) -> str:
    estado = CONSTANTE
    for bloco in blocos(padding(mensagem)):
        estado ^= bloco
        for _ in range(rodadas):
            estado = rotl(estado, 13)
            estado = (estado + CONSTANTE) & MASK64  # soma mod 2^64 mistura bits vizinhos
            estado ^= rotl(estado, 29)
    return f"{estado:016x}"


print("hash_xor_rot('red block blue') =", hash_xor_rot(b"red block blue"))

hash_xor_rot('red block blue') = 76307c9b7a6fe6b3


## 4. Efeito avalanche

Mudamos uma única letra (`red` -> `rad`) e contamos quantos bits do hash mudaram.
Idealmente, cerca de metade dos 64 bits deveria mudar.

In [4]:
def bits_diferentes(h1: str, h2: str) -> int:
    return bin(int(h1, 16) ^ int(h2, 16)).count("1")


for nome, f in [("hash_xor", hash_xor), ("hash_xor_rot", hash_xor_rot)]:
    h1 = f(b"red block blue")
    h2 = f(b"rad block blue")
    print(f"{nome:13s} red: {h1}  rad: {h2}  bits mudados: {bits_diferentes(h1, h2)}/64")

hash_xor      red: 1945064c1709ef13  rad: 1941064c1709ef13  bits mudados: 1/64
hash_xor_rot  red: 76307c9b7a6fe6b3  rad: 0124cdd87a488151  bits mudados: 28/64


## 5. Colisão trivial no XOR puro

Como `XOR` é comutativo, duas mensagens com os mesmos blocos em ordem diferente colidem no `hash_xor`.
Para garantir a troca exata de blocos, construímos as mensagens a partir de dois blocos de 8 bytes.

In [5]:
bloco_a = b"red bloc"   # 8 bytes
bloco_b = b"k blue!!"   # 8 bytes
m1 = bloco_a + bloco_b
m2 = bloco_b + bloco_a
print("m1 =", m1)
print("m2 =", m2)
print()
print("hash_xor    :", hash_xor(m1), hash_xor(m2), "-> colisão!" if hash_xor(m1) == hash_xor(m2) else "")
print("hash_xor_rot:", hash_xor_rot(m1), hash_xor_rot(m2), "-> colisão!" if hash_xor_rot(m1) == hash_xor_rot(m2) else "-> diferentes")

m1 = b'red block blue!!'
m2 = b'k blue!!red bloc'

hash_xor    : 9945064c17094ec2 9945064c17094ec2 -> colisão!
hash_xor_rot: cd99db6bb9d4acbf 151ff30e6de2f336 -> diferentes


## 6. Comparação com o SHA-256

O SHA-256 faz a mesma coisa em espírito (padding, blocos, rotações, XOR e somas), só que com 64 rodadas,
oito palavras de estado, 64 constantes e um tamanho de saída de 256 bits.

In [6]:
import hashlib

print("hash_xor     :", hash_xor(msg))
print("hash_xor_rot :", hash_xor_rot(msg))
print("sha256       :", hashlib.sha256(msg).hexdigest())

hash_xor     : 1945064c1709ef13
hash_xor_rot : 76307c9b7a6fe6b3
sha256       : b0bf36f41c93c6597d2806a34b81cb0703244125d06d33f44a08834b71f30164
